# MP2 Problem 4: Memory and Throughput Experiment

This notebook runs the experiment for **[Problem 4 in README.md](README.md)**. Read the problem statement there for the reference plots, questions, and submission requirements. You can work on this experiment independently of `simple-pytorch.ipynb`; no dataset or pretrained weights are needed.

Use a **Colab GPU, a local CUDA GPU, or a Mac with PyTorch MPS support**. In Colab, choose **Runtime → Change runtime type → GPU**, then upload **`memory_curves.py`** using **Files panel → Upload**. Locally, install `torch`, `torchvision` and `matplotlib`, and keep the script alongside this notebook. For PyTorch installation instructions, see [pytorch.org](https://pytorch.org/get-started/locally/). Upload the script again if Colab resets its files.

Run **Run ResNet-18 memory experiment**, then the results cell. The device check and download cells are optional. `memory_curves.py` is the sole benchmark implementation; the run cell launches that file in a fresh Python process. If the file is elsewhere, set `MEMORY_SCRIPT` to its path.

The default is three complete runs. Use **at least three runs**, as required in README Problem 4. You may reduce the batch-size list if memory is limited. The results table shows `n_runs`, the number of completed runs averaged for each point; follow any warning to reduce batch sizes and rerun.

The last cell optionally downloads the generated figure, averaged and individual CSV measurements, log and environment details. Return to **README Problem 4** to write your answers and comparison.


In [ ]:
#@title Optional: check CUDA/MPS availability
import torch as _mp2_memory_torch

if _mp2_memory_torch.cuda.is_available():
    _mp2_memory_device = "cuda"
    print("GPU:", _mp2_memory_torch.cuda.get_device_name())
elif _mp2_memory_torch.backends.mps.is_available():
    _mp2_memory_device = "mps"
    print("GPU: Apple GPU (MPS); memory maxima are approximate samples.")
else:
    raise RuntimeError(
        "This experiment requires CUDA or Mac MPS. In Colab, select Runtime > Change runtime type > GPU, "
        "then reconnect; or use a Mac with MPS-enabled PyTorch."
    )
print("Device:", _mp2_memory_device)
print("PyTorch:", _mp2_memory_torch.__version__)


In [ ]:
#@title Run ResNet-18 memory experiment
# Upload memory_curves.py in Colab, or keep it alongside this notebook locally.
MEMORY_SCRIPT = "memory_curves.py" #@param {"type":"string"}
# You may reduce this list if your GPU has limited memory.
MEMORY_BATCH_SIZES = [1, 2, 4, 8, 16, 32, 64, 128, 256] #@param {"type":"raw"}
# Use at least three runs for your submitted plot.
MEMORY_RUNS = 3 #@param {"type":"integer"}

def _run_mp2_memory(script, batches, output_root="mp2_memory_results", runs=3):
    import json
    import subprocess
    import sys
    import tempfile
    import time
    from pathlib import Path
    import torch

    script = Path(script).expanduser().resolve()
    if not script.is_file():
        raise FileNotFoundError(
            f"Benchmark script not found: {script}. Upload memory_curves.py using Colab's Files panel, "
            "or place it in the same folder as this notebook locally. Set MEMORY_SCRIPT if it is elsewhere."
        )
    selected = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else None
    if selected is None:
        raise RuntimeError("CUDA or Mac MPS required: in Colab select Runtime > Change runtime type > GPU, or use MPS-enabled PyTorch on a Mac.")
    output_root = Path(output_root)
    output_root.mkdir(parents=True, exist_ok=True)
    run_dir = Path(tempfile.mkdtemp(prefix="run_", dir=output_root))
    command = [sys.executable, "-u", str(script), "--device", selected, "--runs", str(runs),
               "--batches", ",".join(str(batch) for batch in batches), "--out-dir", str(run_dir)]
    started = time.perf_counter()
    with (run_dir / "run.log").open("w") as log:
        process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                   text=True, bufsize=1)
        try:
            for line in process.stdout:
                print(line, end="", flush=True)
                log.write(line)
            returncode = process.wait()
        except BaseException:
            process.terminate()
            try:
                process.wait(timeout=10)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()
            raise
        finally:
            process.stdout.close()
    if returncode:
        raise RuntimeError(f"Memory experiment failed (exit {returncode}). See {run_dir / 'run.log'}.")
    environment = {"device": selected,
                   "gpu": torch.cuda.get_device_name() if selected == "cuda" else "Apple GPU (MPS)",
                   "torch": torch.__version__, "cuda": torch.version.cuda if selected == "cuda" else None,
                   "batch_sizes": batches, "runs": runs,
                   "elapsed_seconds": time.perf_counter() - started}
    (run_dir / "environment.json").write_text(json.dumps(environment, indent=2) + "\n")
    print(f"Results saved in {run_dir}")
    return run_dir


_mp2_memory_run_dir = None
_mp2_memory_run_dir = _run_mp2_memory(MEMORY_SCRIPT, MEMORY_BATCH_SIZES, runs=MEMORY_RUNS)


In [ ]:
def _show_mp2_memory_results(run_dir):
    import csv
    import html
    from pathlib import Path
    from IPython.display import Image, HTML, display

    if run_dir is None:
        raise RuntimeError("Run the experiment cell successfully before displaying results.")
    run_dir = Path(run_dir)
    with (run_dir / "memory_curves_resnet.csv").open() as source:
        rows = list(csv.DictReader(source))
    if not rows:
        raise RuntimeError("No measurements completed. Free GPU/shared memory, reduce batch sizes, or use a fresh Colab GPU session and rerun the experiment.")
    if any(int(row["n_runs"]) < 3 for row in rows):
        print("Some points average fewer than 3 completed runs. Reduce the batch sizes and rerun before submitting.")
    display(Image(filename=str(run_dir / "memory_curves_resnet.png")))
    columns = ["regime", "bs", "n_runs", "max_allocated_MiB", "ms_iter", "img_s"]
    headings = "".join(f"<th>{html.escape(name)}</th>" for name in columns)
    body = []
    for row in rows:
        values = [row["regime"], row["bs"], row["n_runs"]] + [f"{float(row[key]):.2f}" for key in columns[3:]]
        body.append("<tr>" + "".join(f"<td>{html.escape(value)}</td>" for value in values) + "</tr>")
    display(HTML("<table><thead><tr>" + headings + "</tr></thead><tbody>" + "".join(body) + "</tbody></table>"))


_show_mp2_memory_results(globals().get("_mp2_memory_run_dir"))


In [ ]:
# Optional: download the figure, averaged and individual measurements, and run details.
def _download_mp2_memory_results(run_dir):
    import zipfile
    from pathlib import Path
    from IPython.display import FileLink, display

    if run_dir is None:
        raise RuntimeError("Run the experiment cell successfully before downloading results.")
    run_dir = Path(run_dir)
    archive = run_dir / "mp2_memory_results.zip"
    with zipfile.ZipFile(archive, "w", compression=zipfile.ZIP_DEFLATED) as bundle:
        for name in ("memory_curves_resnet.png", "memory_curves_resnet.csv", "memory_curves_resnet_runs.csv", "run.log", "environment.json"):
            bundle.write(run_dir / name, arcname=name)
    try:
        from google.colab import files
    except ImportError:
        display(FileLink(str(archive)))
    else:
        files.download(str(archive))


_download_mp2_memory_results(globals().get("_mp2_memory_run_dir"))
